In [12]:
from huggingface_hub import login
#login("YYY")

In [1]:
!pip install ultralytics datasets opencv-python pillow

  Using cached ultralytics-8.4.33-py3-none-any.whl.metadata (39 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached ultralytics-8.4.33-py3-none-any.whl (1.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 5.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 6.6 MB/s  0:00:00
Using cached hf_xet-1.4.3-cp37

In [11]:
from datasets import load_dataset

dataset = load_dataset("AITC-Traffic-Density/Traffic-Object-Detection")
print(dataset)

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 30
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 30
    })
})


In [3]:
import os

image_dir = "traffic_images"
os.makedirs(image_dir, exist_ok=True)

image_paths = []

for split_name in dataset.keys():
    for i, item in enumerate(dataset[split_name]):
        img = item["image"]
        path = os.path.join(image_dir, f"{split_name}_img_{i}.jpg")
        img.save(path)
        image_paths.append(path)

print(f"Saved {len(image_paths)} images to {image_dir}")
print("First 5 paths:", image_paths[:5])

Saved 160 images to traffic_images
First 5 paths: ['traffic_images/train_img_0.jpg', 'traffic_images/train_img_1.jpg', 'traffic_images/train_img_2.jpg', 'traffic_images/train_img_3.jpg', 'traffic_images/train_img_4.jpg']


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab: Runtime -> Change runtime type -> GPU")

CUDA available: False
No GPU detected. In Colab: Runtime -> Change runtime type -> GPU


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

In [6]:
def count_objects_in_result(result):
    car_count = 0
    pedestrian_count = 0

    if result.boxes is None or result.boxes.cls is None:
        return car_count, pedestrian_count

    for cls_id in result.boxes.cls.tolist():
        cls_id = int(cls_id)
        if cls_id == 0:      # person
            pedestrian_count += 1
        elif cls_id == 2:    # car
            car_count += 1

    return car_count, pedestrian_count

In [7]:
import time
import math
import torch

def benchmark_yolo_gpu_batches(
    model,
    image_paths,
    batch_sizes=(1, 8, 16, 32),
    conf=0.25,
    warmup_batches=2
):
    """
    Benchmark YOLO inference on GPU for multiple batch sizes.

    Measures:
      - total time
      - average latency per image
      - throughput (images/sec)

    Notes:
      - latency per image here is computed as total_time / num_images
      - throughput is num_images / total_time
      - all timing excludes dataset download and file saving
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Enable GPU runtime first.")

    all_results = []

    for batch_size in batch_sizes:
        print(f"\n=== Running batch size {batch_size} ===")

        # Warm-up using first batch
        warmup_sample = image_paths[:min(batch_size, len(image_paths))]
        for _ in range(warmup_batches):
            _ = model.predict(
                source=warmup_sample,
                device="cuda",
                batch=batch_size,
                conf=conf,
                verbose=False
            )
            torch.cuda.synchronize()

        per_image_outputs = []

        total_start = time.perf_counter()

        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]

            batch_start = time.perf_counter()
            results = model.predict(
                source=batch_paths,
                device="cuda",
                batch=batch_size,
                conf=conf,
                verbose=False
            )
            torch.cuda.synchronize()
            batch_end = time.perf_counter()

            batch_time = batch_end - batch_start

            for img_path, result in zip(batch_paths, results):
                cars, pedestrians = count_objects_in_result(result)
                per_image_outputs.append({
                    "image": img_path,
                    "cars": cars,
                    "pedestrians": pedestrians,
                    "batch_size": batch_size,
                    "batch_time_ms": batch_time * 1000.0,
                    "effective_latency_ms": (batch_time / len(batch_paths)) * 1000.0
                })

        total_end = time.perf_counter()
        total_time = total_end - total_start

        num_images = len(image_paths)
        avg_latency_ms = (total_time / num_images) * 1000.0
        throughput = num_images / total_time

        summary = {
            "batch_size": batch_size,
            "num_images": num_images,
            "total_time_s": total_time,
            "avg_latency_ms": avg_latency_ms,
            "throughput_img_s": throughput,
            "outputs": per_image_outputs
        }

        all_results.append(summary)

        print(f"Images processed: {num_images}")
        print(f"Total time:       {total_time:.4f} s")
        print(f"Avg latency:      {avg_latency_ms:.2f} ms/image")
        print(f"Throughput:       {throughput:.2f} images/s")

    return all_results

In [8]:
gpu_results = benchmark_yolo_gpu_batches(
    model=model,
    image_paths=image_paths,
    batch_sizes=(1, 8, 16, 32),
    conf=0.25,
    warmup_batches=2
)

RuntimeError: CUDA is not available. Enable GPU runtime first.

In [ ]:
import pandas as pd

summary_rows = []
for run in gpu_results:
    summary_rows.append({
        "batch_size": run["batch_size"],
        "num_images": run["num_images"],
        "total_time_s": run["total_time_s"],
        "avg_latency_ms": run["avg_latency_ms"],
        "throughput_img_s": run["throughput_img_s"]
    })

gpu_df = pd.DataFrame(summary_rows)
gpu_df

In [ ]:
def benchmark_yolo_cpu(model, image_paths, conf=0.25, warmup_images=3):
    import time

    # Warm-up
    for i in range(min(warmup_images, len(image_paths))):
        _ = model.predict(
            source=image_paths[i],
            device="cpu",
            batch=1,
            conf=conf,
            verbose=False
        )

    outputs = []
    total_start = time.perf_counter()

    for img_path in image_paths:
        img_start = time.perf_counter()
        results = model.predict(
            source=img_path,
            device="cpu",
            batch=1,
            conf=conf,
            verbose=False
        )
        img_end = time.perf_counter()

        cars, pedestrians = count_objects_in_result(results[0])

        outputs.append({
            "image": img_path,
            "cars": cars,
            "pedestrians": pedestrians,
            "latency_ms": (img_end - img_start) * 1000.0
        })

    total_end = time.perf_counter()
    total_time = total_end - total_start
    num_images = len(image_paths)

    return {
        "device": "cpu",
        "batch_size": 1,
        "num_images": num_images,
        "total_time_s": total_time,
        "avg_latency_ms": (total_time / num_images) * 1000.0,
        "throughput_img_s": num_images / total_time,
        "outputs": outputs
    }

In [ ]:
cpu_result = benchmark_yolo_cpu(model, image_paths, conf=0.25)

cpu_df = pd.DataFrame([{
    "device": "cpu",
    "batch_size": cpu_result["batch_size"],
    "num_images": cpu_result["num_images"],
    "total_time_s": cpu_result["total_time_s"],
    "avg_latency_ms": cpu_result["avg_latency_ms"],
    "throughput_img_s": cpu_result["throughput_img_s"]
}])

cpu_df

In [ ]:
gpu_compare_df = gpu_df.copy()
gpu_compare_df["device"] = "gpu"

comparison_df = pd.concat(
    [
        cpu_df[["device", "batch_size", "num_images", "total_time_s", "avg_latency_ms", "throughput_img_s"]],
        gpu_compare_df[["device", "batch_size", "num_images", "total_time_s", "avg_latency_ms", "throughput_img_s"]]
    ],
    ignore_index=True
)

comparison_df = comparison_df.sort_values(["device", "batch_size"]).reset_index(drop=True)
comparison_df

In [ ]:
import matplotlib.pyplot as plt

gpu_plot_df = comparison_df[comparison_df["device"] == "gpu"].copy()

plt.figure(figsize=(8, 5))
plt.plot(gpu_plot_df["batch_size"], gpu_plot_df["throughput_img_s"], marker="o")
plt.xlabel("Batch size")
plt.ylabel("Throughput (images/sec)")
plt.title("GPU Throughput vs Batch Size")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gpu_plot_df["batch_size"], gpu_plot_df["avg_latency_ms"], marker="o")
plt.xlabel("Batch size")
plt.ylabel("Average latency (ms/image)")
plt.title("GPU Average Latency vs Batch Size")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for device in comparison_df["device"].unique():
    df = comparison_df[comparison_df["device"] == device]
    plt.plot(df["batch_size"], df["throughput_img_s"], marker="o", label=device)

plt.xlabel("Batch size")
plt.ylabel("Throughput (images/sec)")
plt.title("CPU vs GPU Throughput Comparison")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for device in comparison_df["device"].unique():
    df = comparison_df[comparison_df["device"] == device]
    plt.plot(df["batch_size"], df["avg_latency_ms"], marker="o", label=device)

plt.xlabel("Batch size")
plt.ylabel("Average latency (ms/image)")
plt.title("CPU vs GPU Latency Comparison")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Get CPU baseline time
cpu_time = comparison_df[comparison_df["device"] == "cpu"]["total_time_s"].values[0]

# Compute speedup
comparison_df["speedup_vs_cpu"] = cpu_time / comparison_df["total_time_s"]

comparison_df

gpu_df = comparison_df[comparison_df["device"] == "gpu"]

plt.figure(figsize=(8, 5))
plt.plot(gpu_df["batch_size"], gpu_df["speedup_vs_cpu"], marker="o")

plt.xlabel("Batch size")
plt.ylabel("Speedup vs CPU")
plt.title("GPU Speedup vs CPU Baseline")
plt.grid(True)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Throughput
for device in comparison_df["device"].unique():
    df = comparison_df[comparison_df["device"] == device]
    axs[0].plot(df["batch_size"], df["throughput_img_s"], marker="o", label=device)

axs[0].set_title("Throughput")
axs[0].set_xlabel("Batch size")
axs[0].set_ylabel("images/sec")
axs[0].grid(True)
axs[0].legend()

# Latency
for device in comparison_df["device"].unique():
    df = comparison_df[comparison_df["device"] == device]
    axs[1].plot(df["batch_size"], df["avg_latency_ms"], marker="o", label=device)

axs[1].set_title("Latency")
axs[1].set_xlabel("Batch size")
axs[1].set_ylabel("ms/image")
axs[1].grid(True)
axs[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
import time
import torch
import pandas as pd

def benchmark_yolo_cpu_batches(
    model,
    image_paths,
    batch_sizes=(1, 8, 16, 32),
    conf=0.25,
    warmup_batches=2
):
    """
    Benchmark YOLO inference on CPU with different batch sizes.
    """

    results = []

    for batch_size in batch_sizes:
        print(f"\n=== CPU batch size {batch_size} ===")

        # Warm-up
        warmup_sample = image_paths[:min(batch_size, len(image_paths))]
        for _ in range(warmup_batches):
            _ = model.predict(
                source=warmup_sample,
                device="cpu",
                batch=batch_size,
                conf=conf,
                verbose=False
            )

        total_start = time.perf_counter()

        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]

            _ = model.predict(
                source=batch_paths,
                device="cpu",
                batch=batch_size,
                conf=conf,
                verbose=False
            )

        total_end = time.perf_counter()
        total_time = total_end - total_start

        num_images = len(image_paths)
        avg_latency_ms = (total_time / num_images) * 1000.0
        throughput = num_images / total_time

        print(f"Images processed: {num_images}")
        print(f"Total time:       {total_time:.4f} s")
        print(f"Avg latency:      {avg_latency_ms:.2f} ms/image")
        print(f"Throughput:       {throughput:.2f} images/sec")

        results.append({
            "device": "cpu",
            "batch_size": batch_size,
            "num_images": num_images,
            "total_time_s": total_time,
            "avg_latency_ms": avg_latency_ms,
            "throughput_img_s": throughput
        })

    return pd.DataFrame(results)


# Run CPU batching benchmark
cpu_batch_df = benchmark_yolo_cpu_batches(
    model=model,
    image_paths=image_paths,
    batch_sizes=(1, 8, 16, 32),
    conf=0.25
)

cpu_batch_df

In [ ]:
cpu_batch_df["device"] = "cpu"
gpu_df["device"] = "gpu"

comparison_full_df = pd.concat([cpu_batch_df, gpu_df], ignore_index=True)
comparison_full_df = comparison_full_df.sort_values(["device", "batch_size"])

comparison_full_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for device in comparison_full_df["device"].unique():
    df = comparison_full_df[comparison_full_df["device"] == device]
    plt.plot(df["batch_size"], df["throughput_img_s"], marker="o", label=device)

plt.xlabel("Batch size")
plt.ylabel("Throughput (images/sec)")
plt.title("CPU vs GPU Throughput (Batching)")
plt.legend()
plt.grid(True)
plt.show()